# Create base year PersonControls file

Creates a PersonControls.csv file for a base year using census and TTS data. The PersonControls file contains the age pyramid by PD, dwelling type and age group.


**Inputs:**

*2021 census profile* downloaded at the aggregated dissimation area level.This data shows all publicly available census totals. While it is also available at the dissimation area level, the ADA was chosen as it relects census tract boundaries but still covers all of Canada. This level is less susceptible to rounding and suppression than the dissemination area level. This data can be downloaded from: https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger.cfm?Lang=E. Since this is a large file, a trimmed version containing only the fields and Aggregated Dissemination Areas in the study area was created using the Notebook 'scripts/population_synthesis/preprocessing/preprocess_census_profile.ipynb'.

*Aggregated dissemination area shapefile* with mapped planning districts. Metrolinx's revised planning district scheme is used as the TTS 2006 planning districts are very large in Durham, Peel and Halton regions. Again, this file has been trimmed using QGIS to only contain ADAs in the model study area.

*TTS zone system shapefile* containing the planning district system.

*TTS age pyramids*, segmented by gender and by people living either in apartments or ground dwellings, downloaded from iDRS.



**Method:**

1. Convert census profile information to wide format and merge into ADA GeoDataFrame file
2. Use areal apportionment to convert census data from ADAs to traffic zones. Then sum to PDs to find age pyramid by gender. 
3. Use the TTS to further segment by dwelling type.
    - Using regional TTS data, find the proportion of people in each age group living in ground units in apartments, by gender.
    - Multiply number of people in each age group, by PD and gender by TTS observed proportion to estimate the number of people in each PD, gender and age group. This is our final answer.
4. Format data as per PersonControls file and write it out.

### Imports

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

from tmg_tdm_tools.common.gis import areal_apportionment
from tmg_tdm_tools.io.dmg_tts import read_tts_cross_tabulation_file

### Constants

In [ ]:
char_ids_to_keep = [
    10, # 	    0 to 4 years
    11, # 	    5 to 9 years
    12, # 	    10 to 14 years
    14, # 	    15 to 19 years
    15, # 	    20 to 24 years
    16, # 	    25 to 29 years
    17, # 	    30 to 34 years
    18, # 	    35 to 39 years
    19, # 	    40 to 44 years
    20, # 	    45 to 49 years
    21, # 	    50 to 54 years
    22, # 	    55 to 59 years
    23, # 	    60 to 64 years
    25,	#      65 to 69 years
    26,	#      70 to 74 years
    27,	#      75 to 79 years
    28,	#      80 to 84 years
    29	#      85 years and over
]
census_gender_cols = {
    'C2_COUNT_MEN+': 'men',
    'C3_COUNT_WOMEN+': 'women'
}
pers_controls_agecats = ["(-Inf,4]", "(4,9]", "(9,14]", "(14,17]", "(17,19]", "(19,24]", "(24,29]", 
                         "(29,34]", "(34,39]", "(39,44]", "(44,54]", "(54,64]", "(64,79]", "(79, Inf]"]
tts_agecat_mapping = {
    '1': "(-Inf,4]",
    '2': "(4,9]",
    '3': "(9,14]",
    '4': "(14,17]",
    '5': "(17,19]",
    '6': "(19,24]",
    '7': "(24,29]",
    '8': "(29,34]",
    '9': "(34,39]",
    '10': "(39,44]",
    '11': "(44,54]",
    '12': "(54,64]",
    '13': "(64,79]",
    '14': "(79, Inf]"
}
tts_region_name_mapping = {
    1: 'Toronto',
    2: 'Durham',
    3: 'York',
    4: 'Peel',
    5: 'Halton',
    6: 'Hamilton',
    11: 'Niagara',
    12: 'Waterloo',
    13: 'Guelph',
    14: 'Wellington',
    15: 'Orangeville',
    16: 'Barrie',
    17: 'Simcoe',
    18: 'Kawartha Lakes',
    19: 'Peterborough City',
    20: 'Peterborough',
    21: 'Orillia',
    22: 'Dufferin',
    23: 'Brantford',
    24: 'Brant'
}

### Inputs

File definitions, file read and initial pre-processing.

In [ ]:
working_dir = [enter working dir here]

In [ ]:
# Census population profile
census_profile_fp = working_dir / "CensusProfiles_98-401-X20210" / "98-401-X2021012_English_CSV_data_trimmed_metropopv1.csv"
census_df = pd.read_csv(
    census_profile_fp, 
    usecols=['ALT_GEO_CODE', 'CHARACTERISTIC_ID', 'C2_COUNT_MEN+', 'C3_COUNT_WOMEN+']
)
census_df = census_df.loc[
    census_df['CHARACTERISTIC_ID'].isin(char_ids_to_keep)].rename(
        census_gender_cols, axis=1).set_index(
            ['ALT_GEO_CODE', 'CHARACTERISTIC_ID'])
# Convert to wide-format for easier processing and eventual merge into ADA GeoDataFrame
census_df = census_df.unstack()

In [ ]:
# Aggregated Dissemination Area (ADA) shapefile
ada_shpfile_fp = working_dir / "GIS/CensusCartographicBoundaryFiles/lada000b21a_e_gtav4zs_epsg2952.shp"
ada_gdf = gpd.read_file(ada_shpfile_fp)
ada_gdf['ADAUID'] = ada_gdf['ADAUID'].astype(np.int64)
ada_gdf = ada_gdf.set_index('ADAUID')
ada_gdf = ada_gdf.drop(['DGUID', 'LANDAREA', 'PRUID'], axis=1)

In [ ]:
# TTS Zone system shapefile, including planning districts in column 'PD2'
tts_zns_fp = working_dir / "GIS/TTS2022ZN/TTS2022zn-withattrs.shp"
tts_gdf = gpd.read_file(tts_zns_fp)
tts_gdf.head()
tts_gdf = tts_gdf.set_index('TTS2022')
tts_gdf = tts_gdf.drop(['PD_no', 'PD_name', 'Mun_no', 'Mun_name', 'Reg_name', 'area', 'perimeter'], axis=1)
tts_gdf = tts_gdf.to_crs(ada_gdf.crs)

In [ ]:
tts_age_pyramid_dir = working_dir / "TTS/2016/AgePyramids/ByRegion"

def read_tts_agepyramid(fp):
    df = read_tts_cross_tabulation_file(
        fp).set_index(
            ['region_hhld', 'age']).unstack(
                ).rename(
                    tts_agecat_mapping, axis=1)
    df.columns = df.columns.droplevel(0)
    df = df.loc[tts_region_name_mapping.values()]
    df.index.name = 'region'
    df.columns.name = 'AgeGroup'
    return df

tts_agepyramid_female = read_tts_agepyramid(tts_age_pyramid_dir / 'ttsAgePyr_Rgn_F.txt')
tts_agepyramid_female_grnd = read_tts_agepyramid(tts_age_pyramid_dir / 'ttsAgePyr_Rgn_Grd_F.txt')
tts_agepyramid_female_apt = read_tts_agepyramid(tts_age_pyramid_dir / 'ttsAgePyr_Rgn_Apt_F.txt')
tts_agepyramid_male = read_tts_agepyramid(tts_age_pyramid_dir / 'ttsAgePyr_Rgn_M.txt')
tts_agepyramid_male_grnd = read_tts_agepyramid(tts_age_pyramid_dir / 'ttsAgePyr_Rgn_Grd_M.txt')
tts_agepyramid_male_apt = read_tts_agepyramid(tts_age_pyramid_dir / 'ttsAgePyr_Rgn_Apt_M.txt')

In [ ]:
output_file = working_dir / "PersonControls.csv"

**1. Process census profile and merge into ADA shapefile**

In [ ]:
# Convert the census age categories into PersonControl file age categories
census_df2_columns = pd.MultiIndex.from_product(
    [census_gender_cols.values(), pers_controls_agecats]
)
census_df2 = pd.DataFrame(index=census_df.index, columns=census_df2_columns)
for gender in census_gender_cols.values():
    census_df2[(gender, "(-Inf,4]")] = census_df[(gender, 10)]
    census_df2[(gender, "(4,9]")] = census_df[(gender, 11)]
    census_df2[(gender, "(9,14]")] = census_df[(gender, 12)]
    census_df2[(gender, "(14,17]")] = 0.6 * census_df[(gender, 14)]    # assume even distribution in this age range
    census_df2[(gender, "(17,19]")] = 0.4 * census_df[(gender, 14)]    # assume even distribution in this age range
    census_df2[(gender, "(19,24]")] = census_df[(gender, 15)]
    census_df2[(gender, "(24,29]")] = census_df[(gender, 16)]
    census_df2[(gender, "(29,34]")] = census_df[(gender, 17)]
    census_df2[(gender, "(34,39]")] = census_df[(gender, 18)]
    census_df2[(gender, "(39,44]")] = census_df[(gender, 19)]
    census_df2[(gender, "(44,54]")] = census_df[(gender, 20)] + census_df[(gender, 21)]
    census_df2[(gender, "(54,64]")] = census_df[(gender, 22)] + census_df[(gender, 23)]
    census_df2[(gender, "(64,79]")] = census_df[(gender, 25)] + census_df[(gender, 26)] + census_df[(gender, 27)]
    census_df2[(gender, "(79, Inf]")] = census_df[(gender, 28)] + census_df[(gender, 29)]

In [ ]:
# Merge census info into the ADA geometries
# We first need to convert the columns to a one-level version to join with the ADA geometries
# as we can't merge dataframes whose columns defined using different number of levels.
census_df2.columns = census_df2.columns.map(':'.join)
ada_gdf2 = ada_gdf.merge(census_df2, left_index=True, right_index=True)

**2. Use areal apportionment to convert census data from ADAs to traffic zones. Then sum to PDs to find age pyramid by gender.**

In [ ]:
# Find the mapping between PDs and regions. 
pd_reg_mapping = tts_gdf[['PD2', 'Reg_no']].drop_duplicates().set_index('PD2').squeeze()
# pd_reg_mapping = pd_reg_mapping.loc[pd_reg_mapping <= 6]
pd_reg_mapping.tail()

In [ ]:
# Now use Areal aportioning to transfer to TAZs
tts_gdf2 = areal_apportionment(ada_gdf2, tts_gdf, columns=census_df2.columns.to_list(), tolerance=0.02)

In [ ]:
pt = tts_gdf2.groupby('PD2')[census_df2.columns].sum()

# Bring the region back in
pt['region'] = pt.index.map(pd_reg_mapping)
# pt = pt.loc[pt['region'] <= 6.0]

# Switch over to region names, not numbers
pt['region'] = pt['region'].map(tts_region_name_mapping)
pt.columns = pt.columns.str.split(':', expand=True)

In [ ]:
# Separate into men and women, keeping the region column so we can do future merges with the TTS data.
pt_men = pd.concat([pt.loc[:, 'men'], pt.loc[:, 'region']], axis=1).rename({np.NaN: "region"}, axis=1)
pt_women = pd.concat([pt.loc[:, 'women'], pt.loc[:, 'region']], axis=1).rename({np.NaN: "region"}, axis=1)

**3. Use the TTS to further segment by dwelling type.**

**Using regional TTS data, find the proportion of people in each age group living in ground units in apartments, by gender.**

In [ ]:
# Separate each gender by ground and apartment
prop_women_grnd = tts_agepyramid_female_grnd / tts_agepyramid_female
prop_women_apt = tts_agepyramid_female_apt / tts_agepyramid_female

prop_men_grnd = tts_agepyramid_male_grnd / tts_agepyramid_male
prop_men_apt = tts_agepyramid_male_apt / tts_agepyramid_male

In [ ]:
# Merge in the relevant proportions to the population by age pivot tables
pt_women_grnd = pt_women.merge(prop_women_grnd, left_on='region', right_index=True, suffixes=['_pop', '_prop'])
pt_women_apt = pt_women.merge(prop_women_apt, left_on='region', right_index=True, suffixes=['_pop', '_prop'])

pt_men_grnd = pt_men.merge(prop_men_grnd, left_on='region', right_index=True, suffixes=['_pop', '_prop'])
pt_men_apt = pt_men.merge(prop_men_apt, left_on='region', right_index=True, suffixes=['_pop', '_prop'])

**Multiply number of people in each age group, by PD and gender by TTS observed proportion to estimate the number of people in each PD, gender and age group. This is our final answer.**

In [ ]:
final_women_grnd = pd.DataFrame(index=pt_women_grnd.index, columns=pers_controls_agecats, dtype=np.float64)
final_women_apt = pd.DataFrame(index=pt_women_apt.index, columns=pers_controls_agecats, dtype=np.float64)
final_men_grnd = pd.DataFrame(index=pt_women_grnd.index, columns=pers_controls_agecats, dtype=np.float64)
final_men_apt = pd.DataFrame(index=pt_women_apt.index, columns=pers_controls_agecats, dtype=np.float64)

for col in pers_controls_agecats:
    final_women_grnd[col] = pt_women_grnd[f'{col}_pop'] * pt_women_grnd[f'{col}_prop']
    final_women_apt[col] = pt_women_apt[f'{col}_pop'] * pt_women_apt[f'{col}_prop']
    final_men_grnd[col] = pt_men_grnd[f'{col}_pop'] * pt_men_grnd[f'{col}_prop']
    final_men_apt[col] = pt_men_apt[f'{col}_pop'] * pt_men_apt[f'{col}_prop']

**4. Format data as per PersonControls file and write it out.**

In [ ]:
final_women_grnd = pd.DataFrame(final_women_grnd.stack())
final_women_grnd.index.names = ['PD', 'AgeGroup']
final_women_grnd.columns = ['Frequency']
final_women_grnd['Sex'] = 'F'
final_women_grnd['DwellingType'] = 1

final_women_apt = pd.DataFrame(final_women_apt.stack())
final_women_apt.index.names = ['PD', 'AgeGroup']
final_women_apt.columns = ['Frequency']
final_women_apt['Sex'] = 'F'
final_women_apt['DwellingType'] = 2

final_men_grnd = pd.DataFrame(final_men_grnd.stack())
final_men_grnd.index.names = ['PD', 'AgeGroup']
final_men_grnd.columns = ['Frequency']
final_men_grnd['Sex'] = 'M'
final_men_grnd['DwellingType'] = 1

final_men_apt = pd.DataFrame(final_men_apt.stack())
final_men_apt.index.names = ['PD', 'AgeGroup']
final_men_apt.columns = ['Frequency']
final_men_apt['Sex'] = 'M'
final_men_apt['DwellingType'] = 2


### Testing

In [ ]:
# Original total population
census_df_sum_sum = census_df.sum().sum()
print(census_df_sum_sum)

In [ ]:
# Number of people after converting to MetroPop age categories
# Should be the exact same
census_df2_sum = census_df2.sum()
assert census_df2_sum.sum() == census_df_sum_sum
print(census_df2_sum.sum())

In [ ]:
# Number of people after merging to ADA geoDataFrame
# Should be the exact same
ada_gdf2_sum = ada_gdf2[census_df2.columns].sum()
assert np.allclose(ada_gdf2_sum, census_df2_sum)
print(ada_gdf2_sum.sum())

In [ ]:
# Number of people after the Areal apportionment to the TTS zones
# Unfortunately we do lost a few people on this, I need to understand why better, 
# Hence my check will be between 0.995 and (1 / 0.995)
tts_gdf2_sum = tts_gdf2[census_df2.columns].sum()
assert (tts_gdf2_sum / ada_gdf2_sum).min() > 0.995
assert (tts_gdf2_sum / ada_gdf2_sum).max() < (1 / 0.995)
print("Total after Areal apportionment:", tts_gdf2_sum.sum())
print("Total difference:", ada_gdf2_sum.sum() - tts_gdf2_sum.sum())

In [ ]:
# Check after segmenting by gender
men_cols = [f"men:{ac}" for ac in pers_controls_agecats]
women_cols = [f"women:{ac}" for ac in pers_controls_agecats]

pt_men_sum = pt_men.drop('region', axis=1).sum()
pt_women_sum = pt_women.drop('region', axis=1).sum()
pt_bygender_sum  = pt_men_sum + pt_women_sum
assert np.allclose(tts_gdf2_sum[men_cols], pt_men_sum)
assert np.allclose(tts_gdf2_sum[women_cols], pt_women_sum)
print("Total after gender segmentation: ", pt_bygender_sum.sum())

In [ ]:
# Check after segment by apartment vs ground dwellings
final_women_grnd_sum = final_women_grnd['Frequency'].unstack().sum()
final_women_apt_sum = final_women_apt['Frequency'].unstack().sum()
final_men_grnd_sum = final_men_grnd['Frequency'].unstack().sum()
final_men_apt_sum = final_men_apt['Frequency'].unstack().sum()
final_sum = final_women_grnd_sum + final_women_apt_sum + final_men_grnd_sum + final_men_apt_sum
assert np.allclose(final_sum, pt_bygender_sum)
print("Totoal population after proportions:", final_sum.sum())

## Testing passed --  Write to file

In [ ]:
final = pd.concat([final_women_grnd, final_men_grnd, final_women_apt, final_men_apt]).reset_index()

# Only keep PDs within the model region
final = final.loc[final['PD'] <= 46]
final = final.set_index(['PD', 'DwellingType', 'Sex', 'AgeGroup'])

In [ ]:
final

In [ ]:
final.to_csv(output_file)